<a href="https://colab.research.google.com/github/RodrigoSampaio4/IA/blob/main/PROJETO_IA_API-GEMINI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# Projeto - Sistema Especialista com Integração Gemini
# Versão pronta para Google Colab
# ============================================================

# 1. Instalação da biblioteca necessária
!pip install -q -U google-genai

# 2. Imports
import os
from getpass import getpass
from google import genai

# 3. Configuração da API Gemini
api_key = getpass("Digite sua GEMINI_API_KEY: ").strip()

if not api_key:
    raise ValueError("Você precisa informar uma GEMINI_API_KEY válida.")

os.environ["GEMINI_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

print("Cliente Gemini configurado com sucesso.")

# 4. Base de conhecimento
BASE_DE_CONHECIMENTO = [
    {
        "id": "R1",
        "descricao": "Se temperatura > 80 e vibração > 70, então alto risco de falha.",
        "condicao": lambda temperatura, vibracao, ruido: temperatura > 80 and vibracao > 70,
        "decisao": "ALTO RISCO DE FALHA",
        "justificativa_tecnica": "Temperatura e vibração estão acima do limite crítico simultaneamente."
    },
    {
        "id": "R2",
        "descricao": "Se temperatura > 60 ou vibração > 50, então risco moderado.",
        "condicao": lambda temperatura, vibracao, ruido: temperatura > 60 or vibracao > 50,
        "decisao": "RISCO MODERADO",
        "justificativa_tecnica": "Pelo menos uma variável ultrapassou o limite de atenção."
    },
    {
        "id": "R3",
        "descricao": "Se ruído > 70, então verificar componentes.",
        "condicao": lambda temperatura, vibracao, ruido: ruido > 70,
        "decisao": "VERIFICAR COMPONENTES",
        "justificativa_tecnica": "Ruído elevado pode indicar desgaste, folga ou desalinhamento."
    },
    {
        "id": "R4",
        "descricao": "Caso contrário, operação normal.",
        "condicao": lambda temperatura, vibracao, ruido: True,
        "decisao": "OPERAÇÃO NORMAL",
        "justificativa_tecnica": "Nenhuma condição crítica foi identificada."
    }
]

print("Base de conhecimento carregada com sucesso.")

# 5. Motor de inferência do sistema especialista
def sistema_especialista(temperatura, vibracao, ruido):
    for regra in BASE_DE_CONHECIMENTO:
        if regra["condicao"](temperatura, vibracao, ruido):
            return {
                "temperatura": temperatura,
                "vibracao": vibracao,
                "ruido": ruido,
                "regra_acionada": regra["id"],
                "descricao_regra": regra["descricao"],
                "decisao": regra["decisao"],
                "justificativa_tecnica": regra["justificativa_tecnica"]
            }

# 6. Função de análise interpretativa com Gemini
def analise_gemini(resultado):
    prompt = f"""
Você é um assistente técnico de apoio à manutenção industrial.

IMPORTANTE:
- Não altere a decisão técnica já tomada pelo sistema especialista.
- Sua função é apenas explicar o resultado em linguagem natural.
- Gere também alertas e sugestões estratégicas.

Dados:
Temperatura: {resultado['temperatura']}
Vibração: {resultado['vibracao']}
Ruído: {resultado['ruido']}

Saída do sistema especialista:
Regra acionada: {resultado['regra_acionada']}
Descrição da regra: {resultado['descricao_regra']}
Decisão técnica: {resultado['decisao']}
Justificativa técnica: {resultado['justificativa_tecnica']}

Responda em 3 partes:
1. Explicação da decisão
2. Alertas estratégicos
3. Sugestões de ajuste
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"Erro na API Gemini: {e}"

# 7. Teste com um caso único
print("\n" + "=" * 60)
print("TESTE COM CASO ÚNICO")
print("=" * 60)

temperatura = 85
vibracao = 75
ruido = 40

resultado = sistema_especialista(temperatura, vibracao, ruido)

print("Decisão do Sistema:", resultado["decisao"])
print("Regra acionada:", resultado["regra_acionada"])
print("Justificativa Técnica:", resultado["justificativa_tecnica"])

resposta_gemini = analise_gemini(resultado)
print("\nGemini:\n")
print(resposta_gemini)

# 8. Testes com múltiplos casos
print("\n" + "=" * 60)
print("TESTES COM MÚLTIPLOS CASOS")
print("=" * 60)

casos_teste = [
    {"temperatura": 85, "vibracao": 75, "ruido": 40},
    {"temperatura": 65, "vibracao": 30, "ruido": 45},
    {"temperatura": 40, "vibracao": 20, "ruido": 80},
    {"temperatura": 35, "vibracao": 25, "ruido": 30},
]

for i, caso in enumerate(casos_teste, start=1):
    resultado = sistema_especialista(
        caso["temperatura"],
        caso["vibracao"],
        caso["ruido"]
    )

    print("=" * 60)
    print(f"CASO {i}")
    print(f"Temperatura: {resultado['temperatura']}")
    print(f"Vibração: {resultado['vibracao']}")
    print(f"Ruído: {resultado['ruido']}")
    print(f"Regra acionada: {resultado['regra_acionada']}")
    print(f"Decisão do sistema: {resultado['decisao']}")
    print(f"Justificativa técnica: {resultado['justificativa_tecnica']}")

    resposta = analise_gemini(resultado)
    print("\nResposta do Gemini:")
    print(resposta)
    print()

# 9. Conclusão
print("=" * 60)
print("CONCLUSÃO")
print("=" * 60)
print("""
Neste projeto foi utilizada a abordagem de Sistema Especialista,
adequada para problemas de diagnóstico baseados em regras rígidas.

A base de conhecimento foi construída com regras SE/ENTÃO,
permitindo ao sistema classificar o estado do equipamento a partir
dos valores de temperatura, vibração e ruído.

Após a tomada de decisão técnica, a API do Gemini foi integrada
para gerar uma análise interpretativa, explicando a decisão em
linguagem natural e sugerindo alertas e ajustes estratégicos.

Esse fluxo separa claramente:
- a decisão técnica, feita pelo sistema especialista;
- a interpretação textual, feita pela IA generativa.
""")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 17.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
Digite sua GEMINI_API_KEY: ··········
Cliente Gemini configurado com sucesso.
Base de conhecimento carregada com sucesso.

TESTE COM CASO ÚNICO
Decisão do Sistema: ALTO RISCO DE FALHA
Regra acionada: R1
Justificativa Técnica: Temperatura e vibração estão acima do limite crítico simultaneamente.

Gemini:

Como seu assistente técnico de apoio à manutenção industrial, analisei os dados e a decisão do sistema especialista. Minha função é traduzir essa informação técnica em lingu